# Molecular GCDG: fresh training and expanded comparison with conditional DiGress

Place this self-contained notebook beside your other notebooks in **Denoise-generator**. It does not require your old GCDG session or trained weights: it trains a new molecular GCDG from the existing exported QM9 graphs. It then evaluates both generators on the exact same held-out condition IDs/vectors. Training and evaluation are enabled below. **Run only when ready to start fresh training.**

The defaults follow `new_copy_84.ipynb`'s molecular configuration, including the QM9 edge-label loss weight 140. They are separate from the synthetic cycle–leg experiment. The provided 49,400-graph development pool and training-fitted frozen NSPPK-14/SVD-128 condition projector are reused as data/preprocessing, not as trained GCDG neural weights. Node-tail SVD is freshly fitted in sparse form before dense padding. New losses, heads and structural controls are trained from scratch.

Conditional DiGress can be evaluated from its existing saved 10,000-condition samples, or resampled from its molecular checkpoint. This notebook does not retrain DiGress automatically. This is a matched evaluation, not an equal-training-budget study. The DiGress exporter supplies the true reference node count; GCDG predicts node count. Report that difference in available information.

Five stages are reported: GCDG before chemical repair, GCDG with native repair, GCDG with a common repair policy, DiGress before chemical repair, and DiGress with that common policy. Common repair uses the same valency limits, downgrade rules and uniform edge priorities for both models. Ties follow node order; the rule is not claimed to be permutation invariant. GCDG native repair additionally uses its learned edge scores. **Raw means after graph construction, before chemical valency repair—not diffusion alone.**

Metrics include matched Morgan similarity and a shuffled-condition control; exact own-condition copying; paired MAE/RMSE/Pearson/Spearman for graph/chemical properties; validity, connectedness, uniqueness, full-reference novelty/copying; nearest-training similarity; internal diversity; BRICS fragment and nonempty Murcko-scaffold frequency similarities; property Wasserstein/MMD; Fréchet ChemNet Distance (FCD); condition-vector preservation; and repair-effect counts. Failed attempts remain in rate denominators. No largest-fragment filter is applied.


In [ ]:
# Active-kernel setup. Install only missing packages; preserve the existing torch build.
import sys, os, importlib, importlib.util, subprocess, tempfile
from pathlib import Path
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select/install a CUDA-enabled PyTorch kernel first; no torch build is chosen here.')
import torch
REQUIRED = {'numpy':'numpy','scipy':'scipy','pandas':'pandas','networkx':'networkx',
    'rdkit':'rdkit','pulp':'PuLP==3.2.2','pytorch_lightning':'pytorch-lightning==2.5.5',
    'torch_geometric':'torch-geometric==2.6.1','matplotlib':'matplotlib','dill':'dill',
    'joblib':'joblib','toolz':'toolz','multiprocessing_on_dill':'multiprocessing-on-dill==3.5.0a4',
    'threadpoolctl':'threadpoolctl','fcd_torch':'fcd_torch'}
missing=[p for m,p in REQUIRED.items() if importlib.util.find_spec(m) is None]
print('Python:',sys.executable)
if missing:
    with tempfile.TemporaryDirectory(prefix='gcdg_molecular_setup_') as t:
        keep=Path(t)/'keep_torch.txt';keep.write_text('torch=='+torch.__version__+'\n')
        subprocess.check_call([sys.executable,'-m','pip','install','--constraint',str(keep),*missing])
    importlib.invalidate_caches()


In [ ]:
from datetime import datetime, timezone
import json, copy, hashlib

def locate_project(override=''):
    start=Path(override).expanduser().resolve() if override else Path.cwd().resolve()
    for p in [start,*start.parents]:
        if (p/'coco_grape').is_dir():return p
    raise FileNotFoundError('Launch Jupyter inside Denoise-generator or set GCDG_PROJECT_ROOT.')
PROJECT_ROOT=locate_project(os.environ.get('GCDG_PROJECT_ROOT',''))
sys.path.insert(0,str(PROJECT_ROOT));sys.dont_write_bytecode=True
OUTPUT_ROOT=Path(os.environ.get('GCDG_MOLECULAR_RUNS',str(Path.home()/'gcdg_molecular_runs'))).expanduser().resolve()
os.environ.setdefault('MPLCONFIGDIR',str(OUTPUT_ROOT/'matplotlib_cache'))
DEVICE=os.environ.get('GCDG_DEVICE','cuda' if torch.cuda.is_available() else 'cpu')
TRAINER_CONFIG=dict(accelerator=DEVICE,devices=1,precision='32-true')
CPU_THREADS = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)

TRAIN_GCDG_FROM_SCRATCH = True
RUN_EVALUATION = True
RESAMPLE_DIGRESS = False       # False re-evaluates the existing matching sample export
TRAINING_SEED=42
TRAIN_GRAPH_LIMIT=None         # None uses all 49,400; a reduced pool is a different experiment
N_CONDITIONS=1000              # pilot; set 10000 for final reporting on the full shared set
SELECTION_SEED=2026
DIGRESS_PYTHON=sys.executable  # set to its own environment's Python if necessary
DIGRESS_CHECKPOINT=None        # required only when RESAMPLE_DIGRESS=True
DIGRESS_SAMPLING_SEED=2026
DIGRESS_BATCH_SIZE=128
DIGRESS_MODEL_LAYERS=9
DIGRESS_EXTRA_FEATURES='all'
PREPARATION_CONFIG=dict(batch_graphs=128,parallel_vectorization=True)

# Files are data exports, not GCDG neural weights. Edit paths if stored elsewhere.
def artifact(name):
    candidates=[PROJECT_ROOT/'DiGress/data'/name,PROJECT_ROOT/'data/qm9_condition_artifacts'/name,
                PROJECT_ROOT/'DiGress/qm9_condition_artifacts'/name]
    return next((p for p in candidates if p.exists()),candidates[0])
TRAIN_EXPORT=artifact('qm9_condition_graphs.pt')
CONDITION_EXPORT=artifact('qm9_condition_decode_inputs.pt')
FROZEN_PROJECTOR=artifact('qm9_nsppk14_svd128_condition_projector.pkl')
DIGRESS_EXPORT=artifact('qm9_condition_generated.pt')
print('Project:',PROJECT_ROOT,'| device:',DEVICE)
print('Outputs:',OUTPUT_ROOT)


In [ ]:
# Molecular hyperparameters extracted from new_copy_84 cell 23.
# No old checkpoint is restored: only the best checkpoint from THIS new fit.
DENOISE_CONFIG = {'latent_embedding_dimension': 128,
 'number_of_transformer_layers': 4,
 'transformer_attention_head_count': 8,
 'transformer_dropout': 0.05,
 'learning_rate': 0.0003,
 'maximum_epochs': 5000,
 'batch_size': 32,
 'total_steps': 50,
 'verbose': True,
 'degree_temperature': None,
 'balance_degree_loss': True,
 'degree_class_weight_cap': 2,
 'lambda_edge_importance': 5,
 'lambda_clean_edge_importance': 0,
 'balance_edge_loss': True,
 'lambda_label_importance': 140,
 'lambda_degree_importance': 60,
 'lambda_node_exist_importance': 5,
 'lambda_consistency': 0.3,
 'lambda_distance_importance': 1,
 'lambda_recon_importance': 0,
 'lambda_x0_importance': 10,
 'lambda_condition_x0_importance': 0,
 'condition_x0_sampling_blend': 0,
 'max_distance_class': 3,
 'label_feature_index': 2,
 'noise_label_factor': 10,
 'noise_degree_factor': 10,
 'sigma_min': 0.01,
 'sigma_max': 0.1,
 'sampling_final_sigma': 0.0,
 'use_guidance': False,
 'early_stopping': True,
 'early_stop_metric': 'val_total',
 'early_stop_mode': 'min',
 'early_stop_patience': 100,
 'early_stop_min_delta': 1e-05,
 'restore_best_weights': True,
 'checkpoint_policy': 'best',
 'use_dim_reduction': True,
 'dim_reduction_method': 'svd',
 'dim_reduction_components': 128,
 'dim_reduction_keep_prefix': 'auto',
 'pca_whiten': False,
 'use_condition_dim_reduction': False,
 'condition_dim_reduction_method': 'svd',
 'condition_dim_reduction_components': 128,
 'condition_dim_reduction_keep_prefix': 0,
 'denoise_discrete_channels': False,
 'discrete_diffusion_mode': 'absorbing',
 'row_embedding_scale': 0.0,
 'discrete_projection_sigma_threshold': 0,
 'use_joint_edge_denoising': True,
 'joint_edge_message_passing_steps': 3,
 'joint_edge_delete_prob': 0.1,
 'joint_edge_add_prob': 0.02,
 'joint_edge_update_momentum': 0.7,
 'joint_edge_threshold': 0.4,
 'random_row_permutation': True,
 'use_matched_row_loss': True,
 'project_existence_during_sampling': True,
 'project_degree_during_sampling': True,
 'project_label_during_sampling': True,
 'use_condition_node_count': True,
 'use_condition_degree_histogram': True,
 'use_condition_label_histogram': True,
 'use_dynamic_molecular_features': True,
 'dynamic_molecular_feature_scale': 1.0,
 'balance_label_loss': True,
 'label_class_weight_cap': 2,
 'lambda_edge_label_importance': 140}
DECODER_CONFIG = {'adjacency_matrix_classifier': None,
 'node_label_classifier': None,
 'edge_label_classifier': None,
 'verbose': True,
 'non_edges_factor': 4,
 'num_augmentation_iterations': 0,
 'augmentation_noise': 0,
 'enforce_connectivity': True,
 'warm_start_mst': True,
 'use_molecular_valency_projection': True,
 'molecular_valency_downgrade_bonds': True,
 'existence_threshold': 0.5,
 'degree_slack_penalty': 20}
PIPELINE_CONFIG = {'use_edge_supervision': True,
 'use_edge_fraction': 0.3,
 'use_distance_supervision': True,
 'distance_edge_blend': 0,
 'edge_generation_mode': 'direct',
 'edge_diffusion_generator': None,
 'verbose': True,
 'use_direct_svd_encodings': False,
 'direct_node_svd_components': 64,
 'direct_graph_svd_components': 64,
 'direct_node_svd_keep_prefix': 3,
 'direct_graph_svd_keep_prefix': 2,
 'direct_svd_random_state': 42}
NODE_PARAMS=dict(radius=1,distance=4,connector=1,nbits=14,dense=False,
                 parallel=PREPARATION_CONFIG['parallel_vectorization'],use_edges_as_features=True)


In [ ]:
EVAL_CONFIG=dict(
    generation_seed=2026,sampling_steps=100,batch_size=10,encoding_batch_size=64,
    metric_seed=2026,fp_radius=2,fp_bits=2048,ignore_stereo=True,
    reference_chemistry='source_smiles', # available reference source SMILES preferred; discrepancies audited
    distribution_cap=2000,diversity_cap=1000,nearest_query_cap=2000,
    compute_fcd=True,fcd_device='cuda:0' if DEVICE=='cuda' else 'cpu',fcd_batch_size=128,
    bootstrap_replicates=500,
    repair_pairs=[('GCDG_pre_repair','GCDG_native_repair'),
                  ('GCDG_pre_repair','GCDG_common_repair'),
                  ('DiGress_pre_repair','DiGress_common_repair')],
    model_pairs=[('GCDG_pre_repair','DiGress_pre_repair'),
                 ('GCDG_common_repair','DiGress_common_repair')],
)
# Optional short software/pipeline PILOT: explicitly a different training budget.
# TRAIN_GRAPH_LIMIT=500
# N_CONDITIONS=100
# DENOISE_CONFIG.update(maximum_epochs=5,early_stop_patience=2)


## Metric definitions and reporting limits

Identity uses canonical heavy-atom SMILES with stereochemistry removed, because these graph representations do not encode stereochemistry. No charge neutralisation, salt stripping or largest-fragment selection is performed. This identity policy can change historical novelty counts and is recorded. Strict graph conversion never invents missing bond labels or performs repair. Reference source SMILES are preferred for chemical descriptors when present; graph-conversion validity/identity disagreements are exported for audit.

Rates use all attempts unless their names explicitly say “over valid”. Paired chemical errors use valid matched pairs; their counts and coverage must accompany results. The all-attempt Tanimoto score assigns zero to unsuccessful pairs and is reported separately from valid-pair similarity. Empty scaffolds are not counted as matching scaffolds; acyclic-pair rates are separate. FCD, distribution and diversity calculations use recorded capped valid subsets and explicitly report their sizes. FCD errors remain unavailable, never replaced by another distance.

These metrics are inspired by [MOSES](https://github.com/molecularsets/moses), with [RDKit Morgan fingerprints](https://www.rdkit.org/docs/source/rdkit.Chem.rdFingerprintGenerator.html) and [fcd_torch](https://github.com/insilicomedicine/fcd_torch). They are not the official MOSES dataset benchmark. In particular fingerprints are 2,048-bit, scaffolds include one-ring systems, and diversity uses the configured cap with duplicates and diagonal pairs retained. Property MMD uses a shared reference-derived bandwidth across stages and is a biased nonnegative estimate; it does not reproduce the old separately bandwidth-selected MMD table. Paired bootstrap intervals describe sampled-condition variation, not variation across training seeds. QED/SA are descriptive metrics on QM9-like small molecules, not evidence of drug suitability.


In [ ]:
"""Sparse preparation for the documented quotient/SVD GCDG configuration.

Node hashes, column order and loss settings are retained. Sparse SVD may differ
numerically from dense SVD; this is a fresh experiment, not bitwise reproduction.
No existing source module or file is changed.
"""
import time
from types import MethodType
import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD


def baseline_quotient_decomposition():
    from coco_grape.module.quotientgraph.operator import (
        add, compose, combination, node, prepend_label, cycle, tree, path,
        neighborhood, unlabel)
    distances = add(*[compose(combination(distance=(d, d)), node(), prepend_label(label=d))
                      for d in range(2, 10)])
    labelled = add(distances, cycle(), tree(), path(number_of_edges=(1, 3)),
                   neighborhood(radius=(0, 2)))
    return add(labelled, compose(labelled, unlabel()))


def inject_sparse_labels(encodings, graphs, label_to_idx=None):
    # Same first-encounter string mapping and insertion at column 2 as source.
    if label_to_idx is None:
        labels = []
        for graph in graphs:
            for _, attrs in graph.nodes(data=True):
                label = attrs.get('label', '__UNLABELED__')
                if label is None or (isinstance(label, str) and not label.strip()):
                    label = '__UNLABELED__'
                labels.append(str(label))
        label_to_idx = {label: i for i, label in enumerate(dict.fromkeys(labels))}
    label_to_idx = {str(k): int(v) for k, v in label_to_idx.items()}
    default = label_to_idx.get('__UNLABELED__', next(iter(label_to_idx.values()), 0))
    out = []
    for x, graph in zip(encodings, graphs):
        x = sparse.csr_matrix(x, dtype=np.float64)
        if x.shape[0] != len(graph) or x.shape[1] < 2:
            raise ValueError('Unexpected quotient feature shape.')
        ids = []
        for _, attrs in graph.nodes(data=True):
            label = attrs.get('label', '__UNLABELED__')
            if label is None or (isinstance(label, str) and not label.strip()):
                label = '__UNLABELED__'
            ids.append(label_to_idx.get(str(label), default))
        labels = sparse.csr_matrix((np.ones(len(ids)), (np.arange(len(ids)), ids)),
                                   shape=(len(ids), len(label_to_idx)))
        out.append(sparse.hstack([x[:, :2], labels, x[:, 2:]], format='csr'))
    return out, label_to_idx


def sparse_generator_class(base):
    """Keep original setup/fit/losses; feed them reduced training arrays only."""
    class SparsePreparedGenerator(base):
        def _reduce_sparse_training(self, matrices):
            out = []
            prefix = self.dim_reduction_keep_prefix
            for start in range(0, len(matrices), self.sparse_batch_graphs):
                batch = matrices[start:start+self.sparse_batch_graphs]
                sizes = [x.shape[0] for x in batch]
                rows = sparse.vstack(batch, format='csr')
                values = np.hstack([rows[:, :prefix].toarray(),
                                    self.reducer.transform(rows[:, prefix:])])
                out.extend(np.split(values, np.cumsum(sizes)[:-1]))
            return out

        def setup(self, node_encodings_list, **kwargs):
            if not all(sparse.issparse(x) for x in node_encodings_list):
                raise ValueError('Memory-efficient setup requires sparse node features.')
            if not self.use_dim_reduction or self.dim_reduction_method != 'svd':
                raise ValueError('Memory-efficient mode requires the baseline node-tail SVD.')
            label_width = int(self._node_label_one_hot_width)
            if getattr(self, '_auto_dim_reduction_keep_prefix', False):
                self.dim_reduction_keep_prefix = 2 + label_width
            prefix = int(self.dim_reduction_keep_prefix)
            started = time.perf_counter()
            print('Sparse node-tail SVD: stacking real training rows (no padded dense array).', flush=True)
            rows = sparse.vstack(node_encodings_list, format='csr')
            self._sparse_raw_width = rows.shape[1]
            self._orig_tail_dim = rows.shape[1] - prefix
            tail = rows[:, prefix:]
            count = max(1, min(int(self.dim_reduction_components),
                               max(1, tail.shape[0]-1), tail.shape[1]))
            if count >= tail.shape[1]:
                raise ValueError('Choose fewer SVD components than hashed tail columns.')
            print(f'Sparse SVD: {tail.shape}, {tail.nnz:,} nonzeros -> {count} components.', flush=True)
            self.reducer = TruncatedSVD(n_components=count, random_state=self.random_state)
            self.reducer.fit(tail)
            self._sparse_reduced_width = prefix + count
            del rows, tail
            reduced = self._reduce_sparse_training(node_encodings_list)
            print(f'Sparse SVD ready in {time.perf_counter()-started:.1f}s; dense width '
                  f'{self._sparse_raw_width} -> {self._sparse_reduced_width}.', flush=True)
            # Original code now pads/scales only the reduced features. The fitted
            # reducer remains attached for raw inference and inverse reconstruction.
            return super().setup(node_encodings_list=reduced, **kwargs)

        def _fit_reducer(self, x):
            if x.shape[-1] != self._sparse_reduced_width or self.reducer is None:
                raise ValueError('Unexpected feature width during sparse preparation.')
            # Already fitted on all real training rows above; do not fit it again.

        def _apply_reducer(self, x):
            if x.shape[-1] == self._sparse_reduced_width:
                return x
            if x.shape[-1] != self._sparse_raw_width:
                raise ValueError('Unexpected inference feature width.')
            return super()._apply_reducer(x)

        def fit(self, node_encodings_list, **kwargs):
            print('Training setup: reduced features ready; fitting condition controls on CPU.', flush=True)
            reduced = self._reduce_sparse_training(node_encodings_list)
            return super().fit(node_encodings_list=reduced, **kwargs)
    return SparsePreparedGenerator


def install_sparse_pipeline(pipeline, batch_graphs=128):
    """Attach sparse encoding and per-graph dense edge sampling to this instance."""
    pipeline.node_graph_vectorizer.return_dense = False
    generator = pipeline.conditioning_to_node_embeddings_generator.conditional_node_generator
    generator.sparse_batch_graphs = int(batch_graphs)
    if generator.sparse_batch_graphs < 1:
        raise ValueError('batch_graphs must be positive.')

    def encode(this, graphs):
        values = []
        for start in range(0, len(graphs), batch_graphs):
            values.extend(this.node_graph_vectorizer.transform(graphs[start:start+batch_graphs]))
            done = min(start+batch_graphs, len(graphs))
            if start == 0 or done == len(graphs) or done % (batch_graphs*8) == 0:
                print(f'Sparse node encoding: {done}/{len(graphs)} graphs', flush=True)
        values, mapping = inject_sparse_labels(values, graphs, getattr(this, '_label_to_idx', None))
        this._label_to_idx = mapping
        this._idx_to_label = {v: k for k, v in mapping.items()}
        this._node_label_one_hot_width = len(mapping)
        decoder = this.node_embeddings_to_graph_generator
        decoder._idx_to_label = this._idx_to_label
        decoder._node_label_one_hot_width = len(mapping)
        generator._idx_to_label = this._idx_to_label
        generator._node_label_one_hot_width = len(mapping)
        conditions = this.graph_encode(graphs)
        decoder._last_conditioning_vectors = list(conditions)
        return values, conditions

    decoder = pipeline.node_embeddings_to_graph_generator
    original_targets = decoder.adj_mtx_to_targets

    def targets(this, adj_mtx_list, node_encodings_list, use_edge_fraction,
                force_bi_directional_edges=True, is_training=False):
        all_targets, all_pairs = [], []
        for gi, (adj, enc) in enumerate(zip(adj_mtx_list, node_encodings_list)):
            # Only one small graph is dense; original distance/order calculations
            # and global random subset selection are retained.
            dense = enc.toarray() if sparse.issparse(enc) else np.asarray(enc)
            y, pairs = original_targets([adj], [dense], use_edge_fraction=1.0,
                                        force_bi_directional_edges=force_bi_directional_edges,
                                        is_training=False)
            all_targets.extend(y.tolist())
            all_pairs.extend((gi, i, j) for _, i, j in pairs)
        if is_training and use_edge_fraction < 1:
            count = int(round(len(all_pairs)*use_edge_fraction))
            indices = np.random.choice(len(all_pairs), count, replace=False)
            all_targets = [all_targets[i] for i in indices]
            all_pairs = [all_pairs[i] for i in indices]
        print(f'Edge supervision ready: {len(all_pairs):,} directed pairs.', flush=True)
        return np.asarray(all_targets), all_pairs

    pipeline.encode = MethodType(encode, pipeline)
    def preserve_sparse(this, values):
        if this.use_direct_svd_encodings:
            raise ValueError('Direct SVD encodings must remain disabled for this sparse baseline path.')
        return values
    # fit() calls this again after encode(); the source implementation casts
    # every array to dense even when direct SVD is disabled.
    pipeline._transform_direct_node_encodings = MethodType(preserve_sparse, pipeline)
    decoder.adj_mtx_to_targets = MethodType(targets, decoder)
    return pipeline


"""Fresh molecular GCDG training from the existing QM9 exports, not old weights."""
from pathlib import Path
from datetime import datetime,timezone
import copy, inspect, json, os, platform, hashlib, subprocess, importlib.metadata
import numpy as np
import networkx as nx
import torch
from scipy import sparse


def pyg_molecular_graphs(payload):
    metadata=payload['metadata'];atoms=metadata['atom_label_values'];bonds=metadata['edge_label_values'];out=[]
    for data in payload['graphs']:
        graph=nx.Graph();labels=torch.as_tensor(data.x).argmax(-1).tolist()
        for i,k in enumerate(labels):graph.add_node(i,label=str(atoms[k]),attr=str(atoms[k]))
        for (u,v),k in zip(torch.as_tensor(data.edge_index).T.tolist(),torch.as_tensor(data.edge_attr).argmax(-1).tolist()):
            if k==0:continue
            if u==v:raise ValueError('Unexpected self loop in molecular training export.')
            label=str(bonds[k])
            if graph.has_edge(u,v) and graph.edges[u,v]['label']!=label:raise ValueError('Conflicting directed training bond labels.')
            graph.add_edge(int(u),int(v),label=label,edge_attr=label)
        graph.graph['export_index']=int(data.idx) if hasattr(data,'idx') else len(out)
        out.append(graph)
    return out


class FrozenMolecularConditions:
    def __init__(self,bundle_path,parallel=False):
        import pickle
        from coco_grape.graph_vectorizer.nsppk import NSPPK
        with Path(bundle_path).open('rb') as f:self.bundle=pickle.load(f)
        params=copy.deepcopy(self.bundle['nsppk_params']);params.update(dense=False,parallel=parallel)
        self.base=NSPPK(**params);self.is_fitted_=False
    def fit(self,graphs,y=None):
        if not self.is_fitted_:self.base.fit(graphs,y);self.is_fitted_=True
        return self
    def transform(self,graphs):
        if not self.is_fitted_:raise RuntimeError('Fit the NSPPK adapter on training graphs first; the SVD remains frozen.')
        raw=self.base.transform(graphs)
        projected=self.bundle['projector'].transform(raw).astype(np.float32)
        return np.asarray((projected-self.bundle['condition_mean'])/self.bundle['condition_std'],dtype=np.float32)


class SparseMolecularNodeVectorizer:
    def __init__(self,params):
        from coco_grape.graph_vectorizer.nsppk import NodeNSPPK
        self.params={**params,'dense':False};self.base=NodeNSPPK(**self.params);self.is_fitted_=False
    def fit(self,graphs,y=None):
        self.base.fit(graphs,y);self.is_fitted_=True;return self
    def transform(self,graphs):
        if not self.is_fitted_:raise RuntimeError('Fit NodeNSPPK on training graphs first.')
        result=[]
        for graph,values in zip(graphs,self.base.transform(graphs)):
            values=sparse.csr_matrix(values,dtype=np.float64)
            if values.shape[0]!=len(graph):raise ValueError('Node feature row count mismatch.')
            prefix=np.array([[1.,graph.degree(u)] for u in graph],dtype=np.float64)
            result.append(sparse.hstack([sparse.csr_matrix(prefix),values],format='csr'))
        return result


def train_molecular_fresh(project_root,output_root,train_graphs,expected_conditions,bundle_path,
                          node_params,denoise_cfg,decoder_cfg,pipeline_cfg,trainer_cfg,preparation_cfg,
                          seed=42,cpu_threads=8):
    import random,time
    from threadpoolctl import threadpool_limits
    import coco_grape.data_processor.generative.conditional_denoising_node_generator as source
    from coco_grape.data_processor.generative.decompositional_encoder_decoder import (
        ConditionalNodeGeneratorModel,DecompositionalNodeEncoderDecoder,DecompositionalEncoderDecoder)
    from pytorch_lightning.loggers import CSVLogger
    accelerator=trainer_cfg['accelerator']
    if accelerator=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA requested but unavailable; no CPU fallback.')
    if accelerator=='mps' and not torch.backends.mps.is_available():raise RuntimeError('MPS requested but unavailable.')
    if accelerator not in ['cuda','mps','cpu']:raise ValueError('Select cuda, mps or cpu.')
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.set_num_threads(cpu_threads)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    run=Path(output_root).expanduser().resolve()/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    run.mkdir(parents=True,exist_ok=False)
    params=copy.deepcopy(denoise_cfg);params['checkpoint_dir']=str(run/'checkpoints')
    unsupported=set(params)-set(inspect.signature(source.ConditionalNodeGenerator.__init__).parameters)
    if unsupported:raise ValueError(f'This generator source does not accept: {unsupported}')
    graph=FrozenMolecularConditions(bundle_path,parallel=bool(preparation_cfg['parallel_vectorization']))
    print('Checking frozen training conditions against the exported values...',flush=True)
    graph.fit(train_graphs)
    check=graph.transform(train_graphs[:min(100,len(train_graphs))])
    expected=np.asarray(expected_conditions)[:len(check)]
    if check.shape!=expected.shape or not np.allclose(check,expected,rtol=1e-4,atol=5e-5):
        raise ValueError('Frozen encoding does not reproduce training-export conditions. Check molecule labels/projector; do not retrain yet.')
    effective_node={**node_params,'parallel':bool(preparation_cfg['parallel_vectorization'])}
    node=SparseMolecularNodeVectorizer(effective_node)
    Generator=sparse_generator_class(source.ConditionalNodeGenerator)
    generator=Generator(**params);generator.device=torch.device(accelerator)
    provider=ConditionalNodeGeneratorModel(conditional_node_generator=generator,verbose=params['verbose'])
    decoder=DecompositionalNodeEncoderDecoder(**copy.deepcopy(decoder_cfg))
    pipeline=DecompositionalEncoderDecoder(graph_vectorizer=graph,node_graph_vectorizer=node,
        conditioning_to_node_embeddings_generator=provider,node_embeddings_to_graph_generator=decoder,**copy.deepcopy(pipeline_cfg))
    install_sparse_pipeline(pipeline,batch_graphs=preparation_cfg['batch_graphs'])
    def save(name,value):(run/name).write_text(json.dumps(value,indent=2,default=str)+'\n')
    save('training_config.json',dict(denoiser=denoise_cfg,decoder=decoder_cfg,pipeline=pipeline_cfg,
        node_features=effective_node,trainer=trainer_cfg,preparation=preparation_cfg,cpu_threads=cpu_threads,
        training_graphs=len(train_graphs),training_seed=seed,from_scratch=True,old_neural_weights_loaded=False,
        frozen_condition_projector=str(bundle_path),frozen_condition_contract_verified=True))
    paths=['coco_grape/data_processor/generative/conditional_denoising_node_generator.py',
           'coco_grape/data_processor/generative/decompositional_encoder_decoder.py']
    save('source_hashes.json',{p:hashlib.sha256((Path(project_root)/p).read_bytes()).hexdigest() for p in paths})
    try:revision=subprocess.check_output(['git','-C',str(project_root),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):revision=None
    versions={}
    for name in ['torch','pytorch-lightning','rdkit','numpy','scipy','scikit-learn','networkx','pulp','fcd_torch']:
        try:versions[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:versions[name]=None
    save('environment.json',dict(python=platform.python_version(),platform=platform.platform(),packages=versions,
        git_revision=revision,accelerator=accelerator,cuda_device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))
    original=source.pl.Trainer
    class ExperimentTrainer(original):
        def __init__(self,*args,**kwargs):
            kwargs.update(trainer_cfg);kwargs.update(default_root_dir=str(run),logger=CSVLogger(str(run),name='training_logs'),
                                                    enable_progress_bar=True,log_every_n_steps=10)
            super().__init__(*args,**kwargs)
        def fit(self,*args,**kwargs):
            # Save the actual optimiser/validation split, when exposed by source loaders.
            for key in ['train_dataloaders','val_dataloaders']:
                loader=kwargs.get(key);indices=getattr(getattr(loader,'dataset',None),'indices',None)
                if indices is not None:save(key+'_indices.json',[int(i) for i in indices])
            return super().fit(*args,**kwargs)
    cwd=Path.cwd();started=time.perf_counter()
    print('Fresh molecular GCDG run:',run,flush=True)
    try:
        os.chdir(run);source.pl.Trainer=ExperimentTrainer
        with threadpool_limits(limits=cpu_threads):pipeline.fit(train_graphs)
    finally:
        source.pl.Trainer=original;os.chdir(cwd)
    save('training_receipt.json',dict(from_scratch=True,old_neural_weights_loaded=False,training_graphs=len(train_graphs),
         fit_seconds=time.perf_counter()-started,run=str(run)))
    # Persist the FULL new pipeline, including reducers and outer structural controls,
    # not just neural weights. Only load this trusted self-created file in future.
    import dill
    trained_generator=pipeline.conditioning_to_node_embeddings_generator.conditional_node_generator
    # Drop Lightning's trainer/data-loader references; preserve a portable CPU snapshot.
    trained_generator.model._trainer=None
    original_device=torch.device(trained_generator.device)
    trained_generator.model.to('cpu');trained_generator.device=torch.device('cpu')
    try:
        with (run/'full_trained_gcdg.pkl').open('wb') as f:dill.dump(pipeline,f,recurse=True)
        save('pipeline_save_status.json',dict(status='complete'))
        print('Full new pipeline saved:',run/'full_trained_gcdg.pkl',flush=True)
    except Exception as exc:
        save('pipeline_save_status.json',dict(status='failed',error=repr(exc)))
        print('Full-pipeline save failed; keep this kernel open:',repr(exc),flush=True)
    finally:
        trained_generator.model.to(original_device);trained_generator.device=original_device
    return pipeline,run


"""Matched GCDG/DiGress molecular evaluation; no model training.

Reference formulas: MOSES (https://github.com/molecularsets/moses).
These are configurable, MOSES-inspired metrics, not the official MOSES dataset
benchmark. Fingerprints: Morgan radius 2, 2048 bits by default.
"""
from pathlib import Path
from collections import Counter
from contextlib import contextmanager
import copy, json, hashlib, time, random, math
import numpy as np
import pandas as pd
import networkx as nx
import torch
from scipy.stats import pearsonr, spearmanr, wasserstein_distance
from scipy.spatial.distance import pdist, cdist
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, QED, Lipinski, rdMolDescriptors, AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
try:
    from rdkit.Contrib.SA_Score import sascorer
except ImportError:
    sascorer = None

BONDS = {'1': Chem.BondType.SINGLE, '1.0': Chem.BondType.SINGLE, 'SINGLE': Chem.BondType.SINGLE,
         '2': Chem.BondType.DOUBLE, '2.0': Chem.BondType.DOUBLE, 'DOUBLE': Chem.BondType.DOUBLE,
         '3': Chem.BondType.TRIPLE, '3.0': Chem.BondType.TRIPLE, 'TRIPLE': Chem.BondType.TRIPLE,
         '1.5': Chem.BondType.AROMATIC, 'AROMATIC': Chem.BondType.AROMATIC}
PROPERTIES = ['atom_count','bond_count','density','mean_degree','carbon_count','nitrogen_count',
              'oxygen_count','fluorine_count','single_bonds','double_bonds','triple_bonds','aromatic_bonds',
              'molecular_weight','logP','QED','TPSA','SA','HBD','HBA','rotatable_bonds',
              'ring_count','aromatic_rings','fraction_csp3']


def json_dump(path, data):
    Path(path).write_text(json.dumps(data, indent=2, default=str)+'\n')


@contextmanager
def instance_override(obj, **values):
    missing=object(); old={k:obj.__dict__.get(k,missing) for k in values}
    for k,v in values.items():setattr(obj,k,v)
    try:yield
    finally:
        for k,v in old.items():
            if v is missing:obj.__dict__.pop(k,None)
            else:setattr(obj,k,v)


def model_digest(module):
    h=hashlib.sha256()
    for name,x in sorted(module.state_dict().items()):
        h.update(name.encode());h.update(x.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def normalise_molecule(mol, ignore_stereo=True):
    mol=Chem.RemoveHs(Chem.Mol(mol))
    if ignore_stereo:Chem.RemoveStereochemistry(mol)
    return mol


def canonical(mol, ignore_stereo=True):
    return Chem.MolToSmiles(normalise_molecule(mol,ignore_stereo),canonical=True,
                            isomericSmiles=not ignore_stereo)


def graph_molecule(graph):
    """Strict conversion: no missing-label defaults, repair, or fragment removal."""
    if graph is None:return None,'missing graph'
    if not len(graph):return None,'empty graph'
    if graph.is_directed() or graph.is_multigraph():return None,'expected simple undirected graph'
    try:
        mol=Chem.RWMol(); ids={}
        for u,d in graph.nodes(data=True):
            if 'label' not in d:raise ValueError(f'missing atom label at {u}')
            atom=str(d['label']).strip()
            if not atom:raise ValueError('empty atom label')
            ids[u]=mol.AddAtom(Chem.Atom(atom))
        for u,v,d in graph.edges(data=True):
            if 'label' not in d:raise ValueError(f'missing bond label on {(u,v)}')
            bond=BONDS.get(str(d['label']).strip().upper())
            if bond is None:raise ValueError(f'unknown bond label {d["label"]!r}')
            mol.AddBond(ids[u],ids[v],bond)
        out=mol.GetMol()
        with rdBase.BlockLogs():Chem.SanitizeMol(out)
        return out,None
    except Exception as exc:return None,str(exc)


def graph_from_projected_arrays(adjacency, node_labels, edge_labels, keep_nodes=None):
    adjacency=np.asarray(adjacency)
    graph=nx.from_numpy_array(adjacency)
    nx.set_node_attributes(graph,{i:str(v) for i,v in enumerate(node_labels)},'label')
    attributes={};k=0
    for i,j in zip(*np.nonzero(adjacency)):
        if k>=len(edge_labels):raise ValueError('edge-label array shorter than directed adjacency entries')
        attributes[(int(i),int(j))]=str(edge_labels[k]);k+=1
    if k!=len(edge_labels):raise ValueError('edge-label count mismatch')
    nx.set_edge_attributes(graph,attributes,'label')
    return graph if keep_nodes is None else graph.subgraph(list(keep_nodes)).copy()


def common_valency_repair(graph, decoder):
    """Same rule for both models; uniform edge priorities and node-order ties."""
    if graph is None:return None
    nodes=list(graph)
    a=nx.to_numpy_array(graph,nodelist=nodes,dtype=int)
    labels=[str(graph.nodes[u]['label']) for u in nodes]
    edge_labels=[str(graph.edges[nodes[i],nodes[j]]['label']) for i,j in zip(*np.nonzero(a))]
    repaired, repaired_labels=decoder._project_molecular_valency(a,labels,np.asarray(edge_labels,dtype=object),prob_matrix=None)
    g=graph_from_projected_arrays(repaired,labels,repaired_labels)
    return nx.relabel_nodes(g,{i:u for i,u in enumerate(nodes)})


def generate_paired_gcdg(pipeline, conditions, cfg, output):
    """One neural/adjacency decode per condition; capture before native repair."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    provider=pipeline.conditioning_to_node_embeddings_generator
    generator=provider.conditional_node_generator;module=generator.model
    decoder=pipeline.node_embeddings_to_graph_generator
    if not hasattr(decoder,'_project_molecular_valency'):
        raise RuntimeError('This decoder version lacks the molecular projection hook.')
    if not bool(getattr(decoder,'use_molecular_valency_projection',False)):
        raise RuntimeError('Native molecular valency projection is disabled in this model. Check the intended molecular configuration.')
    module.eval();before=model_digest(module); raw_graphs=[];native_graphs=[];records=[]
    original=decoder._project_molecular_valency
    seed=int(cfg['generation_seed']);random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    def traced(adj,node_labels,edge_labels,prob_matrix=None):
        trace=dict(adjacency=np.asarray(adj).copy(),node_labels=np.asarray(node_labels).copy(),
                   edge_labels=np.asarray(edge_labels).copy())
        result=original(adj,node_labels,edge_labels,prob_matrix=prob_matrix)
        trace['repaired_adjacency']=np.asarray(result[0]).copy()
        trace['repaired_edge_labels']=np.asarray(result[1]).copy()
        records.append(trace)
        return result
    with instance_override(generator,total_steps=int(cfg['sampling_steps'])), instance_override(decoder,_project_molecular_valency=traced):
        for start in range(0,len(conditions),cfg['batch_size']):
            stop=min(start+cfg['batch_size'],len(conditions));count=len(records)
            with torch.no_grad():generated=pipeline.decode(conditions[start:stop])
            captured=records[count:]
            if len(captured)!=stop-start or len(generated)!=stop-start:
                raise RuntimeError('Projection trace count differs from condition count; comparison is not paired.')
            for graph,trace in zip(generated,captured):
                raw_graphs.append(graph_from_projected_arrays(trace['adjacency'],trace['node_labels'],trace['edge_labels'],graph.nodes()))
                native_graphs.append(graph)
            # Persist completed batches; never discard conditions on failure.
            import pickle
            with (output/'gcdg_paired_partial.pkl').open('wb') as f:
                pickle.dump(dict(raw_graphs=raw_graphs,native_graphs=native_graphs,completed=len(raw_graphs)),f)
            print(f'GCDG matched generation: {stop}/{len(conditions)}',flush=True)
    after=model_digest(module)
    if before!=after:raise RuntimeError('Neural weights changed during evaluation.')
    json_dump(output/'gcdg_pairing.json',dict(weights_unchanged=True,before=before,after=after,
        neural_decode_calls=math.ceil(len(conditions)/cfg['batch_size']),projection_calls=len(records),
        sampling_steps=cfg['sampling_steps'],generation_seed=seed,
        native_projection=True,native_bond_downgrade=bool(decoder.molecular_valency_downgrade_bonds),
        raw_scope='After heads, controls and constrained adjacency, before chemical valency repair'))
    return raw_graphs,native_graphs


def digress_graphs(payload, metadata):
    atom_labels=metadata['atom_label_values'];edge_labels=metadata['edge_label_values'];out=[]
    for sample in payload['samples']:
        atoms,edges=[torch.as_tensor(x).detach().cpu() for x in sample]
        if atoms.ndim==2:atoms=atoms.argmax(-1)
        if edges.ndim==3:edges=edges.argmax(-1)
        if atoms.ndim!=1 or edges.shape!=(len(atoms),len(atoms)):
            raise ValueError('Unexpected DiGress sample shape.')
        if not torch.equal(edges,edges.T):raise ValueError('Asymmetric DiGress sample: inspect conversion rather than silently fixing.')
        graph=nx.Graph()
        for i,a in enumerate(atoms.tolist()):
            a=int(a)
            if not 0<=a<len(atom_labels):raise ValueError('Unknown DiGress atom class.')
            graph.add_node(i,label=str(atom_labels[a]))
        for i in range(len(atoms)):
            for j in range(i+1,len(atoms)):
                e=int(edges[i,j])
                if e==0:continue
                if not 0<e<len(edge_labels):raise ValueError('Unknown DiGress edge class.')
                if str(edge_labels[e])=='__no_edge__':continue
                graph.add_edge(i,j,label=str(edge_labels[e]))
        out.append(graph)
    return out


def validate_shared_payloads(inputs, digress):
    x=np.asarray(inputs['condition_y']);y=np.asarray(digress['condition_y'])
    if x.shape!=y.shape or not np.allclose(x,y,rtol=1e-5,atol=1e-5):
        raise ValueError('DiGress output conditions/order do not match the shared evaluation payload.')
    if len(inputs['condition_graphs_nx'])!=len(digress['samples']) or len(x)!=len(digress['samples']):
        raise ValueError('Condition/sample counts differ.')
    if len(inputs['targets'])!=len(set(inputs['targets'])):
        raise ValueError('Condition IDs are not unique.')
    if list(inputs['targets'])!=list(digress['targets']):raise ValueError('DiGress target IDs/order differ.')
    nodes=digress.get('n_nodes')
    return dict(condition_vectors_and_order_match=True,n_conditions=len(x),
                digress_supplied_reference_node_counts=nodes is not None,
                condition_sha256=hashlib.sha256(np.asarray(x,dtype=np.float32).tobytes()).hexdigest())


def normalise_train_smiles(strings, cfg):
    result=set();failures=[]
    for value in strings:
        with rdBase.BlockLogs():mol=Chem.MolFromSmiles(str(value))
        if mol is None:failures.append(str(value))
        else:result.add(canonical(mol,cfg['ignore_stereo']))
    if failures:raise ValueError(f'{len(failures)} invalid training-reference SMILES: repair the reference before computing novelty.')
    return result


def molecule_row(graph, index, cfg, reference=False):
    mol,error=graph_molecule(graph);graph_mol=mol
    graph_smiles=canonical(mol,cfg['ignore_stereo']) if mol is not None else None
    chemistry_source='graph'
    # Source SMILES are ground truth for reference chemistry when available.
    # Keep graph-conversion disagreements in a separate audit instead of hiding them.
    if reference and cfg['reference_chemistry']=='source_smiles' and graph is not None and graph.graph.get('smiles'):
        with rdBase.BlockLogs():source=Chem.MolFromSmiles(str(graph.graph['smiles']))
        if source is not None:mol=source;chemistry_source='source_smiles'
    row=dict(index=index,valid=mol is not None,conversion_error=error,
             graph_conversion_valid=graph_mol is not None,chemistry_source=chemistry_source,
             graph_smiles=graph_smiles,smiles=canonical(mol,cfg['ignore_stereo']) if mol is not None else None,
             connected=bool(graph is not None and len(graph)>0 and nx.is_connected(graph)))
    row['reference_graph_identity_agrees']=row['smiles']==graph_smiles if reference else None
    for feature in PROPERTIES:row[feature]=np.nan
    if graph is not None:
        n=len(graph);e=graph.number_of_edges();atoms=Counter(str(d.get('label')) for _,d in graph.nodes(data=True))
        bonds=Counter(str(d.get('label')).upper() for _,_,d in graph.edges(data=True))
        row.update(atom_count=n,bond_count=e,density=2*e/(n*(n-1)) if n>1 else 0,mean_degree=2*e/n if n else 0,
            carbon_count=atoms['C'],nitrogen_count=atoms['N'],oxygen_count=atoms['O'],fluorine_count=atoms['F'],
            single_bonds=sum(bonds[k] for k in ['1','1.0','SINGLE']),
            double_bonds=sum(bonds[k] for k in ['2','2.0','DOUBLE']),
            triple_bonds=sum(bonds[k] for k in ['3','3.0','TRIPLE']),aromatic_bonds=sum(bonds[k] for k in ['1.5','AROMATIC']))
    row['_mol']=normalise_molecule(mol,cfg['ignore_stereo']) if mol is not None else None
    row['scaffold']=None
    if mol is not None:
        mol=row['_mol']
        row.update(molecular_weight=Descriptors.MolWt(mol),logP=Descriptors.MolLogP(mol),QED=QED.qed(mol),
                   TPSA=Descriptors.TPSA(mol),HBD=Lipinski.NumHDonors(mol),HBA=Lipinski.NumHAcceptors(mol),
                   rotatable_bonds=Lipinski.NumRotatableBonds(mol),ring_count=rdMolDescriptors.CalcNumRings(mol),
                   aromatic_rings=rdMolDescriptors.CalcNumAromaticRings(mol),fraction_csp3=rdMolDescriptors.CalcFractionCSP3(mol))
        if sascorer is not None:row['SA']=sascorer.calculateScore(mol)
        scaffold=MurckoScaffold.MurckoScaffoldSmiles(mol=mol,includeChirality=False)
        row['scaffold']=scaffold if scaffold else None
    return row


def safe_correlation(x,y,method='pearson'):
    if len(x)<2 or np.std(x)<1e-12 or np.std(y)<1e-12:return np.nan
    return float((pearsonr if method=='pearson' else spearmanr)(x,y)[0])


def counter_cosine(a,b):
    if not a or not b:return np.nan
    return sum(a[k]*b[k] for k in a.keys() & b.keys()) / math.sqrt(sum(v*v for v in a.values())*sum(v*v for v in b.values()))


def molecular_fragments(mols):
    counts=Counter()
    for mol in mols:counts.update(Chem.MolToSmiles(AllChem.FragmentOnBRICSBonds(mol)).split('.'))
    return counts


def biased_mmd(x,y,bandwidth,batch=256):
    x=np.asarray(x).reshape(-1,1);y=np.asarray(y).reshape(-1,1)
    def kernel_mean(a,b):
        total=0.
        for start in range(0,len(a),batch):
            total+=np.exp(-cdist(a[start:start+batch],b,'sqeuclidean')/(2*bandwidth**2)).sum()
        return total/(len(a)*len(b))
    return float(max(0.,kernel_mean(x,x)+kernel_mean(y,y)-2*kernel_mean(x,y)))


def evaluate_comparison(reference_graphs, condition_ids, stages, training_smiles, cfg, output,
                        vector_encoder=None, condition_vectors=None):
    """Score matched attempts and export explicit denominators and failure rows."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    n=len(reference_graphs)
    if n<2 or not stages:raise ValueError('Provide at least two conditions and at least one generator stage.')
    if min(cfg['distribution_cap'],cfg['diversity_cap'])<2 or min(cfg['nearest_query_cap'],cfg['bootstrap_replicates'])<1:
        raise ValueError('Distribution/diversity caps must be >=2; nearest-neighbour queries and bootstrap replicates must be positive.')
    if len(condition_ids)!=n or len(set(condition_ids))!=n:raise ValueError('Invalid condition IDs.')
    if any(len(g)!=n for g in stages.values()):raise ValueError('Every stage must retain one attempt per condition.')
    train=normalise_train_smiles(training_smiles,cfg)
    if not train:raise ValueError('Full training reference is required for novelty.')
    references=[molecule_row(g,i,cfg,True) for i,g in enumerate(reference_graphs)]
    rows={};rates=[];property_stats=[];vector_stats=[];vector_pairs=[];repair_stats=[];issues=[]
    if sascorer is None:issues.append(dict(stage='all',metric='SA',error='RDKit Contrib SA scorer unavailable.'))
    if vector_encoder is None:issues.append(dict(stage='all',metric='condition_vectors',error='No frozen encoder supplied.'))
    if vector_encoder is not None and condition_vectors is None:raise ValueError('Supply the matched frozen condition vectors.')
    fingerprint=rdFingerprintGenerator.GetMorganGenerator(radius=cfg['fp_radius'],fpSize=cfg['fp_bits'])
    fp_cache={}
    def fp(smiles,mol):
        if smiles not in fp_cache:fp_cache[smiles]=fingerprint.GetFingerprint(mol)
        return fp_cache[smiles]
    reference_fps=[fp(r['smiles'],r['_mol']) if r['valid'] else None for r in references]
    rng=np.random.default_rng(cfg['metric_seed']);shift=int(rng.integers(1,n)) if n>1 else 0
    permuted=np.roll(np.arange(n),shift) # deranged for n>1; saved to reproduce control
    for stage,graphs in stages.items():
        print('Molecular metrics:',stage,flush=True)
        current=[molecule_row(g,i,cfg) for i,g in enumerate(graphs)];rows[stage]=current
        for i,(row,ref) in enumerate(zip(current,references)):
            row.update(stage=stage,condition_id=condition_ids[i],train_copy=bool(row['valid'] and row['smiles'] in train),
                       novel=bool(row['valid'] and row['smiles'] not in train))
            row['own_condition_copy']=bool(row['valid'] and ref['valid'] and row['smiles']==ref['smiles'])
            row['paired_chemistry_valid']=bool(row['valid'] and ref['valid'])
            value=DataStructs.TanimotoSimilarity(fp(row['smiles'],row['_mol']),reference_fps[i]) if row['paired_chemistry_valid'] else np.nan
            row['paired_tanimoto']=value
            row['paired_tanimoto_invalid_zero']=value if np.isfinite(value) else 0.
            other=reference_fps[permuted[i]]
            row['shuffled_condition_tanimoto']=DataStructs.TanimotoSimilarity(fp(row['smiles'],row['_mol']),other) if row['valid'] and other is not None and n>1 else np.nan
            row['matched_tanimoto_shuffle_eligible']=value if np.isfinite(value) and np.isfinite(row['shuffled_condition_tanimoto']) else np.nan
            row['scaffold_match']=bool(row['scaffold']==ref['scaffold']) if row['scaffold'] and ref['scaffold'] else np.nan
            row['both_acyclic']=bool(row['valid'] and ref['valid'] and row['scaffold'] is None and ref['scaffold'] is None)
            for feature in PROPERTIES:
                row[feature+'_abs_error']=abs(row[feature]-ref[feature]) if np.isfinite(row[feature]) and np.isfinite(ref[feature]) else np.nan
        valid=[r for r in current if r['valid']];unique={r['smiles'] for r in valid};unique_novel=unique-train
        paired=[r for r in current if r['paired_chemistry_valid']];scaffold_rows=[r for r in current if pd.notna(r['scaffold_match'])]
        summary=dict(stage=stage,attempts=n,valid_count=len(valid),validity=len(valid)/n,
                     connected_rate=sum(r['connected'] for r in current)/n,
                     valid_connected_rate=sum(r['valid'] and r['connected'] for r in current)/n,
                     unique_valid=len(unique),uniqueness_over_valid=len(unique)/len(valid) if valid else np.nan,
                     train_copies=sum(r['train_copy'] for r in current),
                     train_copy_rate_over_valid=sum(r['train_copy'] for r in current)/len(valid) if valid else np.nan,
                     novelty_over_valid=sum(r['novel'] for r in current)/len(valid) if valid else np.nan,
                     valid_unique_novel_rate=len(unique_novel)/n,
                     own_condition_copy_rate_over_attempts=sum(r['own_condition_copy'] for r in current)/n,
                     paired_valid_count=len(paired),paired_tanimoto_mean=np.mean([r['paired_tanimoto'] for r in paired]) if paired else np.nan,
                     paired_tanimoto_invalid_zero_mean=np.mean([r['paired_tanimoto_invalid_zero'] for r in current]),
                     shuffled_tanimoto_mean=pd.Series([r['shuffled_condition_tanimoto'] for r in current if np.isfinite(r['matched_tanimoto_shuffle_eligible'])],dtype=float).mean(),
                     matched_tanimoto_shuffle_eligible_mean=pd.Series([r['matched_tanimoto_shuffle_eligible'] for r in current]).mean(),
                     matched_shuffle_pair_count=sum(np.isfinite(r['matched_tanimoto_shuffle_eligible']) for r in current),
                     nonempty_scaffold_pairs=len(scaffold_rows),
                     scaffold_preservation=np.mean([r['scaffold_match'] for r in scaffold_rows]) if scaffold_rows else np.nan,
                     both_acyclic_rate=sum(r['both_acyclic'] for r in current)/n,
                     generated_nonempty_scaffold_fraction=sum(bool(r['scaffold']) for r in valid)/len(valid) if valid else np.nan)
        for k in [1000,10000]:
            first=valid[:k]
            summary[f'unique_at_{k}']=len({r['smiles'] for r in first})/len(first) if len(first)==k else np.nan
            summary[f'unique_at_{k}_eligible']=len(first)==k
        rates.append(summary)
        for feature in PROPERTIES:
            indices=[i for i,(r,c) in enumerate(zip(current,references)) if np.isfinite(r[feature]) and np.isfinite(c[feature])]
            x=np.array([references[i][feature] for i in indices]);y=np.array([current[i][feature] for i in indices])
            property_stats.append(dict(stage=stage,property=feature,n_pairs=len(indices),mae=np.mean(abs(y-x)) if len(x) else np.nan,
                rmse=np.sqrt(np.mean((y-x)**2)) if len(x) else np.nan,pearson=safe_correlation(x,y),spearman=safe_correlation(x,y,'spearman'),
                exact_match_rate=np.mean(np.isclose(x,y,rtol=0,atol=1e-8)) if len(x) else np.nan,
                generated_mean=np.mean(y) if len(x) else np.nan,condition_mean=np.mean(x) if len(x) else np.nan))
        if vector_encoder is not None:
            indices=[i for i,g in enumerate(graphs) if g is not None and len(g)>0]
            if indices:
                encoded=[]
                for start in range(0,len(indices),cfg['encoding_batch_size']):
                    encoded.extend(np.asarray(vector_encoder([graphs[i] for i in indices[start:start+cfg['encoding_batch_size']]])))
                y=np.asarray(encoded);x=np.asarray(condition_vectors)[indices]
                if x.shape!=y.shape:raise ValueError('Frozen condition-vector dimensions differ.')
                if not np.isfinite(x).all() or not np.isfinite(y).all():raise ValueError('Nonfinite graph encoding.')
                cosine=np.divide((x*y).sum(1),np.linalg.norm(x,axis=1)*np.linalg.norm(y,axis=1),out=np.full(len(x),np.nan),where=np.linalg.norm(x,axis=1)*np.linalg.norm(y,axis=1)>1e-12)
                correlations=[safe_correlation(x[:,d],y[:,d]) for d in range(x.shape[1])]
                vector_stats.append(dict(stage=stage,n_pairs=len(indices),dimensions=x.shape[1],active_dimensions=sum(np.isfinite(correlations)),
                    mean_same_dim_pearson=np.nanmean(correlations) if np.isfinite(correlations).any() else np.nan,
                    mean_pair_cosine=np.nanmean(cosine),mean_l2=np.mean(np.linalg.norm(x-y,axis=1))))
                for j,i in enumerate(indices):vector_pairs.append(dict(stage=stage,condition_id=condition_ids[i],cosine=cosine[j],l2=np.linalg.norm(x[j]-y[j])))
            else:issues.append(dict(stage=stage,metric='condition_vectors',error='No nonempty generated graphs.'))
        pd.DataFrame([{k:v for k,v in r.items() if k!='_mol'} for stage_rows in rows.values() for r in stage_rows]).to_csv(output/'per_molecule.csv',index=False)
    # Use the same capped valid sample size across stages for distribution/diversity metrics.
    ref_valid=[r for r in references if r['valid']]
    common=min([len(ref_valid),cfg['distribution_cap'],*[sum(r['valid'] for r in rr) for rr in rows.values()]])
    if common<2:issues.append(dict(stage='all',metric='distribution',error='Fewer than two common valid molecules.'))
    distributions=[];diversities=[];fcd_rows=[];sample_ids={}
    ref_indices=rng.choice(len(ref_valid),common,replace=False) if common else []
    ref_sample=[ref_valid[i] for i in ref_indices]
    for stage,current in rows.items():
        valid=[r for r in current if r['valid']]
        chosen=rng.choice(len(valid),common,replace=False) if common else []
        sample=[valid[i] for i in chosen];sample_ids[stage]=[r['index'] for r in sample]
        if common>=2:
            for feature in PROPERTIES:
                x=np.array([r[feature] for r in ref_sample]);y=np.array([r[feature] for r in sample])
                x=x[np.isfinite(x)];y=y[np.isfinite(y)]
                if len(x)<2 or len(y)<2:continue
                bandwidth_values=pdist(x[:min(500,len(x))].reshape(-1,1));positive=bandwidth_values[bandwidth_values>0]
                bandwidth=float(np.median(positive)) if len(positive) else 1.
                distributions.append(dict(stage=stage,property=feature,reference_n=len(x),generated_n=len(y),
                    wasserstein1=wasserstein_distance(x,y),mmd2_biased=biased_mmd(x,y,bandwidth),bandwidth=bandwidth))
            mols=[r['_mol'] for r in sample];ref_mols=[r['_mol'] for r in ref_sample]
            drows=sample[:min(cfg['diversity_cap'],len(sample))];fps=[fp(r['smiles'],r['_mol']) for r in drows]
            sums=squares=0.
            for f in fps:
                values=np.asarray(DataStructs.BulkTanimotoSimilarity(f,fps));sums+=values.sum();squares+=(values**2).sum()
            scaffolds=Counter(r['scaffold'] for r in sample if r['scaffold']);ref_scaffolds=Counter(r['scaffold'] for r in ref_sample if r['scaffold'])
            diversities.append(dict(stage=stage,diversity_n=len(fps),intdiv1=1-sums/len(fps)**2,
                intdiv2=1-math.sqrt(squares/len(fps)**2),fragment_frequency_cosine=counter_cosine(molecular_fragments(mols),molecular_fragments(ref_mols)),
                scaffold_frequency_cosine_min_rings1=counter_cosine(scaffolds,ref_scaffolds),
                unique_nonempty_scaffolds=len(scaffolds),scaffold_eligible_generated=sum(scaffolds.values()),scaffold_eligible_reference=sum(ref_scaffolds.values())))
    # Nearest neighbour to FULL training reference, evaluated on a recorded query subset.
    train_fps=[fp(s,Chem.MolFromSmiles(s)) for s in sorted(train)]
    nearest=[]
    for stage,current in rows.items():
        valid=[r for r in current if r['valid']]
        count=min(len(valid),cfg['nearest_query_cap']);chosen=rng.choice(len(valid),count,replace=False) if count else []
        similarities=[]
        for k in chosen:
            row=valid[k];value=max(DataStructs.BulkTanimotoSimilarity(fp(row['smiles'],row['_mol']),train_fps))
            similarities.append(value);nearest.append(dict(stage=stage,condition_id=condition_ids[row['index']],nearest_train_tanimoto=value))
        for summary in rates:
            if summary['stage']==stage:
                summary.update(nearest_train_query_n=count,nearest_train_reference_n=len(train),
                    nearest_train_tanimoto_mean=np.mean(similarities) if count else np.nan,
                    near_train_analogue_rate_0_9=np.mean(np.asarray(similarities)>=.9) if count else np.nan)
        print('Nearest-training similarity ready:',stage,flush=True)
    if cfg['compute_fcd'] and common>=2:
        try:
            from fcd_torch import FCD
            # fcd_torch 1.x loads its own bundled ChemNet pickle. Scope the torch
            # 2.6 compatibility to this installed package's known model file only.
            import importlib
            fcd_module=importlib.import_module(FCD.__module__);package_root=Path(fcd_module.__file__).resolve().parent
            class TorchCompatibility:
                def __getattr__(self,key):return getattr(torch,key)
                def load(self,path,*args,**kwargs):
                    if Path(path).resolve().parent==package_root and Path(path).name=='ChemNet_v0.13_pretrained.pt':
                        kwargs.setdefault('weights_only',False)
                    return torch.load(path,*args,**kwargs)
            with instance_override(fcd_module,torch=TorchCompatibility()):
                fcd=FCD(device=cfg['fcd_device'],n_jobs=1,batch_size=cfg['fcd_batch_size'])
            reference_stats=fcd.precalc([r['smiles'] for r in ref_sample])
            for stage,indices in sample_ids.items():
                values=[rows[stage][i]['smiles'] for i in indices]
                fcd_rows.append(dict(stage=stage,n_generated=len(values),n_reference=common,
                                     fcd=float(fcd(gen=values,pref=reference_stats)),status='complete'))
        except Exception as exc:
            issues.append(dict(stage='all',metric='FCD',error=repr(exc)))
            for stage in stages:fcd_rows.append(dict(stage=stage,fcd=np.nan,status='unavailable: '+repr(exc)))
    else:
        for stage in stages:fcd_rows.append(dict(stage=stage,fcd=np.nan,status='disabled' if not cfg['compute_fcd'] else 'insufficient common valid molecules'))
    # Repair effects: paired changes in structure, validity and connectedness.
    for before,after in cfg['repair_pairs']:
        if before not in rows or after not in rows:continue
        b=rows[before];a=rows[after];removed=downgraded=changed=0
        for bg,ag in zip(stages[before],stages[after]):
            if bg is None or ag is None:continue
            br={frozenset((u,v)):str(d['label']) for u,v,d in bg.edges(data=True)}
            ar={frozenset((u,v)):str(d['label']) for u,v,d in ag.edges(data=True)}
            removed+=len(br.keys()-ar.keys());downgraded+=sum(br[k]!=ar[k] for k in br.keys() & ar.keys())
            changed+=br!=ar
        repair_stats.append(dict(before=before,after=after,attempts=n,changed_graphs=changed,edges_removed=removed,
            bond_labels_changed=downgraded,invalid_to_valid=sum(not r['valid'] and s['valid'] for r,s in zip(b,a)),
            valid_to_invalid=sum(r['valid'] and not s['valid'] for r,s in zip(b,a)),
            connected_to_disconnected=sum(r['connected'] and not s['connected'] for r,s in zip(b,a))))
    # Paired bootstrap across conditions: not uncertainty across retrainings.
    differences=[]
    for first,second in cfg.get('model_pairs',[]):
        if first not in rows or second not in rows:continue
        for metric in ['valid','connected','paired_tanimoto_invalid_zero']+[p+'_abs_error' for p in ['molecular_weight','logP','QED','TPSA']]:
            x=np.asarray([r[metric] for r in rows[first]],float);y=np.asarray([r[metric] for r in rows[second]],float)
            valid=np.isfinite(x)&np.isfinite(y);delta=x[valid]-y[valid]
            if not len(delta):continue
            means=[np.mean(delta[rng.integers(0,len(delta),len(delta))]) for _ in range(cfg['bootstrap_replicates'])]
            differences.append(dict(first=first,second=second,metric=metric,n_pairs=len(delta),difference=np.mean(delta),
                ci95_low=np.quantile(means,.025),ci95_high=np.quantile(means,.975),uncertainty='condition-pair bootstrap; one generation draw per condition'))
    frames={'summary':pd.DataFrame(rates),'paired_properties':pd.DataFrame(property_stats),
            'condition_vectors':pd.DataFrame(vector_stats),'vector_pairs':pd.DataFrame(vector_pairs),
            'distribution_properties':pd.DataFrame(distributions),'diversity':pd.DataFrame(diversities),
            'nearest_training':pd.DataFrame(nearest),'fcd':pd.DataFrame(fcd_rows),'repair_effects':pd.DataFrame(repair_stats),
            'paired_model_differences':pd.DataFrame(differences),'issues':pd.DataFrame(issues,columns=['stage','metric','error'])}
    for name,frame in frames.items():frame.to_csv(output/(name+'.csv'),index=False)
    pd.DataFrame([{k:v for k,v in r.items() if k!='_mol'} for r in references]).to_csv(output/'reference_conversion_audit.csv',index=False)
    json_dump(output/'evaluation_config.json',dict(config=cfg,full_training_unique_smiles=len(train),
        distribution_common_valid_n=common,reference_distribution_indices=[r['index'] for r in ref_sample],
        stage_distribution_indices=sample_ids,shuffled_condition_indices=permuted.tolist(),
        SA_available=sascorer is not None,condition_vector_encoder_available=vector_encoder is not None,
        identity_policy='heavy-atom, canonical SMILES; stereo ignored' if cfg['ignore_stereo'] else 'canonical isomeric SMILES',
        no_largest_fragment_filter=True,no_chemical_repair_in_conversion=True,
        diversity_definition='1-E[T] and 1-sqrt(E[T^2]); diagonal and duplicate outputs retained in capped sample',
        scaffold_definition='nonempty Bemis-Murcko scaffolds; one-ring scaffolds included (MOSES default is two rings)'))
    return frames


def plot_compact(frames, output):
    import matplotlib.pyplot as plt
    s=frames['summary'].set_index('stage')
    labels=[x.replace('_',' ') for x in s.index]
    fig,axes=plt.subplots(1,3,figsize=(11,3.4))
    for ax,metric,title in zip(axes,['validity','connected_rate','paired_tanimoto_invalid_zero_mean'],
                              ['RDKit validity','Connected graphs','Matched-condition similarity']):
        values=s[metric].to_numpy();ax.barh(labels,values,color=['#639dc0','#245570','#9cb5c4','#dc9d60','#9d6435'][:len(s)])
        ax.set_xlim(0,1.03);ax.set_title(title);ax.spines[['top','right']].set_visible(False)
        for j,v in enumerate(values):ax.text(v+.01,j,f'{v:.3f}',va='center',fontsize=8)
        if ax is not axes[0]:ax.set_yticklabels([])
    fig.tight_layout();fig.savefig(Path(output)/'molecular_comparison.png',dpi=220,bbox_inches='tight');plt.show()
    columns=['validity','connected_rate','uniqueness_over_valid','novelty_over_valid',
             'train_copy_rate_over_valid','valid_unique_novel_rate','paired_tanimoto_invalid_zero_mean']
    table=s[columns].T
    table.to_csv(Path(output)/'compact_table.csv')
    table.to_latex(Path(output)/'compact_table.tex',float_format='%.3f',escape=True)
    return table


## Load and check the shared exports

The `.pt` and `.pkl` files below must be your own trusted exports. Git may not transfer these large files; copy them separately to the remote machine if absent. No data or model is downloaded automatically. The full training-reference SMILES set comes from export metadata, not a 10k subset.


In [ ]:
for p in [TRAIN_EXPORT,CONDITION_EXPORT,FROZEN_PROJECTOR]:
    if not p.is_file():raise FileNotFoundError(f'Missing export: {p}. Copy the original artifact to the remote host or edit its path.')
training_payload=torch.load(TRAIN_EXPORT,map_location='cpu',weights_only=False)
shared_inputs=torch.load(CONDITION_EXPORT,map_location='cpu',weights_only=False)
selected_training_payload=training_payload if TRAIN_GRAPH_LIMIT is None else {**training_payload, 'graphs':training_payload['graphs'][:int(TRAIN_GRAPH_LIMIT)]}
TRAIN_GRAPHS=pyg_molecular_graphs(selected_training_payload)
EXPECTED_TRAIN_CONDITIONS=np.asarray(training_payload['condition_y'])[:len(TRAIN_GRAPHS)]
metadata=shared_inputs['metadata']
FULL_TRAIN_SMILES=metadata.get('train_smiles_set_full') or training_payload['metadata'].get('train_smiles_set_full')
if not FULL_TRAIN_SMILES:raise RuntimeError('Missing full training-reference canonical SMILES set.')
# Fail before expensive training if the reference or saved DiGress comparison is incompatible.
normalise_train_smiles(FULL_TRAIN_SMILES,EVAL_CONFIG)
if not RESAMPLE_DIGRESS:
    if not DIGRESS_EXPORT.is_file():raise FileNotFoundError(f'Missing DiGress samples: {DIGRESS_EXPORT}')
    _digress_preflight=torch.load(DIGRESS_EXPORT,map_location='cpu',weights_only=False)
    validate_shared_payloads(shared_inputs,_digress_preflight)
    del _digress_preflight
# The frozen transform contains no GCDG neural weights; no SVD refit is performed here.
probe=FrozenMolecularConditions(FROZEN_PROJECTOR,parallel=PREPARATION_CONFIG['parallel_vectorization']).fit(TRAIN_GRAPHS)
if not 1 < N_CONDITIONS <= len(shared_inputs['condition_graphs_nx']):raise ValueError('Choose 2..available conditions.')
selection=np.random.default_rng(SELECTION_SEED).choice(len(shared_inputs['condition_graphs_nx']),N_CONDITIONS,replace=False)
REFERENCES=[shared_inputs['condition_graphs_nx'][i] for i in selection]
CONDITION_IDS=[shared_inputs['targets'][i] for i in selection]
CONDITIONS=np.asarray(shared_inputs['condition_y'])[selection]
recomputed=probe.transform(REFERENCES[:min(100,len(REFERENCES))])
if not np.allclose(recomputed,CONDITIONS[:len(recomputed)],rtol=1e-4,atol=5e-5):
    raise RuntimeError('Shared condition vectors do not match the frozen projector. Do not compare generators yet.')
print('Training graphs:',len(TRAIN_GRAPHS),'| selected conditions:',len(REFERENCES),
      '| full development-reference SMILES:',len(FULL_TRAIN_SMILES))
print('Frozen condition contract verified.')


## Fresh molecular GCDG training

No previous GCDG model is required. The original 5,000-epoch cap and validation early stopping remain editable above; full training can take a long time. Sparse preparation removes the previous giant padded feature allocation, but does not eliminate training costs. The original internal 10% validation split is retained and its actual indices are saved. This is distinct from DiGress's original 70/10/20 internal split: report training scopes and budgets rather than claiming they are matched.

The complete new pipeline (neural model, reducers and auxiliary controls) is saved, as well as Lightning checkpoints. A pipeline-save failure is recorded explicitly; keep this kernel open for evaluation. No previous-run resume path is included.


In [ ]:
if TRAIN_GCDG_FROM_SCRATCH:
    import pulp
    if not pulp.PULP_CBC_CMD(msg=False).available():raise RuntimeError('CBC solver is not available.')
    MOLECULAR_MODEL,RUN_DIR=train_molecular_fresh(
        PROJECT_ROOT,OUTPUT_ROOT,TRAIN_GRAPHS,EXPECTED_TRAIN_CONDITIONS,FROZEN_PROJECTOR,
        NODE_PARAMS,DENOISE_CONFIG,DECODER_CONFIG,PIPELINE_CONFIG,TRAINER_CONFIG,PREPARATION_CONFIG,
        seed=TRAINING_SEED,cpu_threads=CPU_THREADS)
    json_dump(RUN_DIR/'selection.json',dict(condition_ids=CONDITION_IDS,export_indices=selection.tolist(),
        selection_seed=SELECTION_SEED,evaluation_conditions=N_CONDITIONS,
        training_pool_graphs=len(TRAIN_GRAPHS),common_reference_scope='full original 49400-graph development pool; includes DiGress internal validation/test',
        source_preset='new_copy_84.ipynb cell 23 (QM9 edge-label weight=140)',
        historical_result_reproduction=False))
else:
    print('Fresh training disabled. Evaluation requires MOLECULAR_MODEL from a fresh fit in this kernel.')


## Optional DiGress resampling (no retraining)

The existing export can be scored with all new metrics. To draw new samples, set `RESAMPLE_DIGRESS=True` and provide the intended molecular checkpoint. Its model/config settings must match that checkpoint. Use the DiGress environment's Python if it differs. A child process preserves the original sampler script, explicitly moves the loaded model to the chosen device and permits loading your trusted pickle-containing input exports. All outputs go to the new run; the existing export is retained. No DiGress training is started here.


In [ ]:
if RESAMPLE_DIGRESS:
    if DIGRESS_CHECKPOINT is None or not Path(DIGRESS_CHECKPOINT).expanduser().is_file():
        raise FileNotFoundError('Set DIGRESS_CHECKPOINT to the intended molecular .ckpt.')
    if 'RUN_DIR' not in globals():raise RuntimeError('Run fresh GCDG training first to create a run directory.')
    DIGRESS_REPO=PROJECT_ROOT/'DiGress'
    sampler=RUN_DIR/'run_digress_sampler.py'
    # Original script remains unmodified. This wrapper only configures loading/device in its child process.
    wrapper = "import os,sys,runpy\nfrom pathlib import Path\nimport torch\nroot=Path(os.environ['GCDG_DIGRESS_REPO']);sys.path.insert(0,str(root));os.chdir(root)\ndevice=os.environ['GCDG_DIGRESS_DEVICE']\nif device=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA unavailable in DiGress Python.')\noriginal_load=torch.load\ndef trusted_load(*args,**kwargs):\n    kwargs.setdefault('weights_only',False)\n    return original_load(*args,**kwargs)\ntorch.load=trusted_load\nfrom src.diffusion_model_discrete import DiscreteDenoisingDiffusion\noriginal_checkpoint=DiscreteDenoisingDiffusion.load_from_checkpoint\n@classmethod\ndef loaded_on_device(cls,*args,**kwargs):\n    return original_checkpoint(*args,**kwargs).to(device).eval()\nDiscreteDenoisingDiffusion.load_from_checkpoint=loaded_on_device\ntorch.manual_seed(int(os.environ['GCDG_DIGRESS_SEED']))\nrunpy.run_path(str(root/'src/sample_qm9_condition.py'),run_name='__main__')\n"
    sampler.write_text(wrapper)
    resampled=RUN_DIR/'digress_generated_all_conditions.pt'
    command=[str(DIGRESS_PYTHON),str(sampler),'dataset=qm9_condition','general.wandb=disabled',
        f'general.test_only="{Path(DIGRESS_CHECKPOINT).expanduser().resolve()}"',
        f'dataset.conditions_file="{CONDITION_EXPORT.resolve()}"',
        f'dataset.conditioned_output_file="{resampled.resolve()}"',
        f'train.batch_size={DIGRESS_BATCH_SIZE}',f'model.n_layers={DIGRESS_MODEL_LAYERS}',
        f'model.extra_features={DIGRESS_EXTRA_FEATURES}']
    env=os.environ.copy();env.update(PYTHONPATH=str(DIGRESS_REPO),GCDG_DIGRESS_REPO=str(DIGRESS_REPO),
        GCDG_DIGRESS_DEVICE=DEVICE,GCDG_DIGRESS_SEED=str(DIGRESS_SAMPLING_SEED),
        DIGRESS_QM9_SKIP_TRANSFORMER_MASK='1',OMP_NUM_THREADS=str(CPU_THREADS))
    with (RUN_DIR/'digress_sampling.log').open('w') as log:
        process=subprocess.Popen(command,cwd=DIGRESS_REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:print(line,end='');log.write(line);log.flush()
        if process.wait()!=0:raise RuntimeError('DiGress sampling failed; inspect digress_sampling.log.')
    DIGRESS_EXPORT=resampled


## Matched generation and expanded evaluation

GCDG is generated once per selected condition. A hook captures adjacency and labels before its native valency projection, so the native before/after comparison shares the neural sample and constrained graph construction. The common repair is applied to each model's raw graph using identical unweighted priorities. Sample counts/condition IDs/vectors are checked before scoring. The full reference and condition projector remain fixed.

Generation batches and per-molecule evaluation rows are saved. The first 1,000 conditions are a pilot by default; set `N_CONDITIONS=10000` for the full comparison. Distribution metrics use up to 2,000 valid molecules per stage with a common reference sample; increasing that cap increases MMD/FCD costs. Nearest-neighbour queries are capped but search the full canonical training reference; query IDs are exported. FCD's large-scale empirical reliability and runtime should be considered when selecting its sample size.


In [ ]:
if RUN_EVALUATION:
    if 'MOLECULAR_MODEL' not in globals():raise RuntimeError('Train the fresh molecular model above first.')
    if not DIGRESS_EXPORT.is_file():raise FileNotFoundError(f'Missing DiGress samples: {DIGRESS_EXPORT}')
    dg=torch.load(DIGRESS_EXPORT,map_location='cpu',weights_only=False)
    pairing=validate_shared_payloads(shared_inputs,dg)
    evaluation=RUN_DIR/'molecular_comparison';evaluation.mkdir(parents=True,exist_ok=False)
    json_dump(evaluation/'shared_pairing.json',pairing)
    measured=np.asarray(MOLECULAR_MODEL.graph_encode(REFERENCES))
    if not np.allclose(measured,CONDITIONS,rtol=1e-4,atol=5e-5):raise RuntimeError('New GCDG condition encoder differs from the shared contract.')
    raw_gcdg,native_gcdg=generate_paired_gcdg(MOLECULAR_MODEL,CONDITIONS,EVAL_CONFIG,evaluation)
    digress_all=digress_graphs(dg,shared_inputs['metadata'])
    raw_digress=[digress_all[i] for i in selection]
    from coco_grape.data_processor.generative.decompositional_encoder_decoder import DecompositionalNodeEncoderDecoder
    common_decoder=DecompositionalNodeEncoderDecoder(adjacency_matrix_classifier=None,node_label_classifier=None,
        edge_label_classifier=None,use_molecular_valency_projection=True,molecular_valency_downgrade_bonds=True,verbose=False)
    STAGES={'GCDG_pre_repair':raw_gcdg,'GCDG_native_repair':native_gcdg,
            'GCDG_common_repair':[common_valency_repair(g,common_decoder) for g in raw_gcdg],
            'DiGress_pre_repair':raw_digress,
            'DiGress_common_repair':[common_valency_repair(g,common_decoder) for g in raw_digress]}
    import pickle
    with (evaluation/'matched_graphs.pkl').open('wb') as f:
        pickle.dump(dict(condition_ids=CONDITION_IDS,condition_graphs=REFERENCES,condition_vectors=CONDITIONS,stages=STAGES),f)
    RESULTS=evaluate_comparison(REFERENCES,CONDITION_IDS,STAGES,FULL_TRAIN_SMILES,EVAL_CONFIG,evaluation,
                               vector_encoder=MOLECULAR_MODEL.graph_encode,condition_vectors=CONDITIONS)
    display(plot_compact(RESULTS,evaluation))
    display(RESULTS['paired_properties']);display(RESULTS['diversity']);display(RESULTS['fcd'])
    display(RESULTS['repair_effects']);display(RESULTS['paired_model_differences'])
    if not RESULTS['issues'].empty:display(RESULTS['issues'])
    print('Saved results:',evaluation)
else:
    print('Evaluation disabled.')


## What to include in the thesis

Use the compact validity/diversity/novelty table together with paired property errors, paired molecular similarity and vector preservation. Report both the valid-pair counts and failure-aware rates. Use `repair_effects.csv` to quantify how many graphs changed, became valid, lost connectedness, lost edges or changed bond labels. Preserve the native-versus-common repair distinction. Do not call chemically repaired graphs raw neural outputs. Include reference-conversion disagreements and missing metric/error statuses in your audit.

FCD/fragment/scaffold/property-distribution agreement measures aggregate similarity, not preservation of a particular input. Pair-specific Tanimoto and property errors address the latter. Compare matched similarity with its shuffled-condition control. High nearest-training similarity can indicate close analogues even when exact-SMILES novelty is high. Do not treat QED or SA as measures of therapeutic effectiveness. Repeat training/sampling seeds if you need conclusions beyond this one trained run; bootstrap intervals here do not provide that uncertainty.

The old thesis MMD table is not overwritten or silently combined with the new estimator. Keep fresh-training results separate until checked. Git tracks this notebook; runs/checkpoints stay outside the checkout. Existing generator and thesis files are unchanged.
